# Cross-Domain Review Sentiment Classification

This notebook implements Option 2 (Review Sentiment Analysis) for ARTI 404. It trains a binary sentiment classifier (Positive/Negative) on Amazon product and IMDb movie reviews, then evaluates it on an in-domain held-out test and on a cross-domain Yelp restaurant test.

Experimental question: which method performs better on restaurant reviews when trained on product and movie reviews? The main judgment is Macro F1 on the Yelp test, with Accuracy reported alongside; the in-domain test is a supporting comparison only. Two methods are compared: TF-IDF + Logistic Regression (Method 1) and frozen MiniLM sentence embeddings + Logistic Regression (Method 2). Settings for both methods are selected on Development data only; neither test set is used for tuning.

## Setup

Import the libraries used across the notebook.

In [1]:
import platform
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sentence_transformers
import sklearn
import torch

c:\Users\moham\Desktop\nlp project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fix the random seed so splits, shuffling and model fitting are reproducible.

In [2]:
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

Define relative project paths and create the output folders if they do not already exist. The notebook is expected to run from the project root, so every path here is relative rather than a path tied to one member's machine.

In [3]:
PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RESULTS = PROJECT_ROOT / "results"
MODELS = PROJECT_ROOT / "models"

for path in (DATA_RAW, DATA_PROCESSED, RESULTS, MODELS):
    path.mkdir(parents=True, exist_ok=True)

Record the exact package versions used in this run, so the environment can be reproduced later.

In [4]:
print("Python:", platform.python_version())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("torch:", torch.__version__)

Python: 3.12.10
numpy: 2.5.4
pandas: 3.1.1
scikit-learn: 1.9.1
sentence-transformers: 6.2.0
torch: 2.15.0+cpu
